# HingBERT text classification
Fine-tune `l3cube-pune/hing-bert` on the processed Hinglish dataset. Run processing.ipynb first, then run these cells in order. The first model load requires internet access. A CUDA GPU is recommended; CPU training is supported but slower.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "transformers", "pandas", "scikit-learn", "matplotlib", "tqdm"])


In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/dataset/processed_dataset.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run processing.ipynb first and open this notebook inside the project.")
DATA_DIR = ROOT / "data/dataset"
OUTPUT_DIR = ROOT / "models/hingbert"
CHECKPOINT = "l3cube-pune/hing-bert"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
print("Device:", DEVICE)


In [ ]:
df = pd.read_csv(DATA_DIR / "processed_dataset.csv")
label2id = json.loads((DATA_DIR / "label_mapping.json").read_text(encoding="utf-8"))
id2label = {int(v): k for k, v in label2id.items()}
assert sorted(id2label) == list(range(len(id2label)))
assert not df[["text", "label_id", "split"]].isna().any().any()
assert set(df["split"]) == {"train", "test"}
assert (df["label"].map(label2id) == df["label_id"]).all()
train_pool = df.loc[df["split"] == "train"].copy()
test_df = df.loc[df["split"] == "test"].reset_index(drop=True)
train_df, val_df = train_test_split(train_pool, test_size=0.1, stratify=train_pool["label_id"], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print({"train": len(train_df), "validation": len(val_df), "test": len(test_df)})
print(pd.DataFrame({name: frame.label.value_counts(normalize=True) for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]}))


In [ ]:
# Disable Xet acceleration before importing/downloading hub files.
# Restart the kernel before running all cells if huggingface_hub was already imported.
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
from huggingface_hub import snapshot_download

try:
    checkpoint_path = snapshot_download(
        repo_id=CHECKPOINT,
        allow_patterns=["config.json", "model.safetensors", "tokenizer.json",
                        "tokenizer_config.json", "special_tokens_map.json", "vocab.txt"],
    )
except Exception:
    import traceback
    traceback.print_exc(chain=True)
    raise

tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint_path, use_safetensors=True, num_labels=len(label2id), id2label=id2label, label2id=label2id
).to(DEVICE)
# Newly initialized classifier weights are expected: training learns this head.
class TextDataset(Dataset):
    def __init__(self, frame):
        self.encodings = tokenizer(frame["text"].tolist(), truncation=True, max_length=MAX_LENGTH)
        self.labels = frame["label_id"].astype(int).tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        item["labels"] = self.labels[index]
        return item

collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
train_loader = DataLoader(TextDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(TextDataset(val_df), batch_size=BATCH_SIZE, collate_fn=collator)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    targets, predictions = [], []
    total_loss = 0.0
    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item() * len(batch["labels"])
        targets.extend(batch["labels"].cpu().tolist())
        predictions.extend(outputs.logits.argmax(dim=-1).cpu().tolist())
    return {"loss": total_loss / len(targets), "accuracy": accuracy_score(targets, predictions),
            "macro_f1": f1_score(targets, predictions, labels=list(id2label), average="macro", zero_division=0)}, targets, predictions

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_f1 = -1.0
history = []
import time
from datetime import timedelta

training_started = time.perf_counter()
batch_seconds = 0.0
validation_seconds = 0.0
validations_completed = 0
completed_batches = 0
total_batches = EPOCHS * len(train_loader)

def duration(seconds):
    return str(timedelta(seconds=max(0, round(seconds))))

print("Training ETA appears after the first batch; validation estimates improve after epoch 1.")
overall_progress = tqdm(total=total_batches, desc="Overall training", unit="batch")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch_started = time.perf_counter()
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += outputs.loss.item() * len(batch["labels"])
        # loss.item() above synchronizes GPU work before measuring elapsed time.
        batch_seconds += time.perf_counter() - batch_started
        completed_batches += 1
        remaining_batches = total_batches - completed_batches
        remaining_validations = EPOCHS - validations_completed
        validation_eta = (validation_seconds / validations_completed * remaining_validations
                          if validations_completed else 0.0)
        eta = batch_seconds / completed_batches * remaining_batches + validation_eta
        overall_progress.update(1)
        overall_progress.set_postfix(
            epoch=f"{epoch + 1}/{EPOCHS}",
            remaining=duration(eta),
            elapsed=duration(time.perf_counter() - training_started),
            validation="included" if validations_completed else "pending estimate",
        )
    validation_started = time.perf_counter()
    metrics, _, _ = evaluate(val_loader)
    validation_seconds += time.perf_counter() - validation_started
    validations_completed += 1
    record = {"epoch": epoch + 1, "train_loss": total_loss / len(train_df), **metrics}
    history.append(record)
    print(record)
    if metrics["macro_f1"] > best_f1:
        best_f1 = metrics["macro_f1"]
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        (OUTPUT_DIR / "label_mapping.json").write_text(json.dumps(label2id, indent=2), encoding="utf-8")
pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

overall_progress.close()
print(f"Training completed in {duration(time.perf_counter() - training_started)}")


Evaluate the best validation checkpoint on the test set after training. Avoid using test results to select hyperparameters.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(DEVICE)
test_loader = DataLoader(TextDataset(test_df), batch_size=BATCH_SIZE, collate_fn=collator)
test_metrics, targets, predictions = evaluate(test_loader)
print(test_metrics)
print(classification_report(targets, predictions, labels=list(id2label), target_names=list(id2label.values()), zero_division=0))
(OUTPUT_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")
ConfusionMatrixDisplay.from_predictions(targets, predictions, labels=list(id2label), display_labels=list(id2label.values()), xticks_rotation=45, cmap="Blues")
plt.tight_layout()
plt.show()


In [ ]:
@torch.no_grad()
def predict(text):
    model.eval()
    inputs = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
    probabilities = model(**inputs).logits.softmax(dim=-1)[0]
    index = int(probabilities.argmax().item())
    return {"label": model.config.id2label[index], "score": float(probabilities[index].item())}

predict("Aaj ka din bahut accha tha")
